# Anomaly Detection in System Logs Using Autoencoders

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

def generate_synthetic_log_data(n_samples=10000):
    data = []

    log_types = ['INFO', 'WARNING', 'ERROR', 'DEBUG']
    services = ['web_server', 'database', 'auth_service', 'file_server', 'api_gateway']

    for i in range(n_samples):
        if np.random.random() < 0.95:
            log_type = np.random.choice(log_types, p=[0.6, 0.25, 0.1, 0.05])
            service = np.random.choice(services)
            response_time = np.random.normal(200, 50) if log_type != 'ERROR' else np.random.normal(1000, 200)
            cpu_usage = np.random.normal(30, 10) if log_type != 'ERROR' else np.random.normal(80, 15)
            memory_usage = np.random.normal(40, 15) if log_type != 'ERROR' else np.random.normal(85, 10)
            request_size = np.random.normal(1024, 300)
            is_anomaly = 0
        else:
            log_type = np.random.choice(['ERROR', 'WARNING'], p=[0.7, 0.3])
            service = np.random.choice(services)
            response_time = np.random.normal(5000, 1000)
            cpu_usage = np.random.normal(95, 5)
            memory_usage = np.random.normal(95, 5)
            request_size = np.random.normal(10000, 2000)
            is_anomaly = 1

        data.append({
            'log_type': log_type,
            'service': service,
            'response_time': max(0, response_time),
            'cpu_usage': np.clip(cpu_usage, 0, 100),
            'memory_usage': np.clip(memory_usage, 0, 100),
            'request_size': max(0, request_size),
            'is_anomaly': is_anomaly
        })

    return pd.DataFrame(data)

def preprocess_data(df):
    df_processed = df.copy()

    le_log_type = LabelEncoder()
    le_service = LabelEncoder()

    df_processed['log_type_encoded'] = le_log_type.fit_transform(df_processed['log_type'])
    df_processed['service_encoded'] = le_service.fit_transform(df_processed['service'])

    feature_columns = ['log_type_encoded', 'service_encoded', 'response_time',
                      'cpu_usage', 'memory_usage', 'request_size']

    scaler = StandardScaler()
    df_processed[feature_columns] = scaler.fit_transform(df_processed[feature_columns])

    return df_processed, feature_columns, scaler, le_log_type, le_service

def build_autoencoder(input_dim, encoding_dim=8):
    input_layer = Input(shape=(input_dim,))

    encoded = Dense(32, activation='relu')(input_layer)
    encoded = Dropout(0.2)(encoded)
    encoded = Dense(16, activation='relu')(encoded)
    encoded = Dropout(0.2)(encoded)
    encoded = Dense(encoding_dim, activation='relu')(encoded)

    decoded = Dense(16, activation='relu')(encoded)
    decoded = Dropout(0.2)(decoded)
    decoded = Dense(32, activation='relu')(decoded)
    decoded = Dropout(0.2)(decoded)
    decoded = Dense(input_dim, activation='linear')(decoded)

    autoencoder = Model(input_layer, decoded)
    encoder = Model(input_layer, encoded)

    return autoencoder, encoder

print("Generating synthetic log data...")
df = generate_synthetic_log_data(10000)

print("Dataset shape:", df.shape)
print("\nData overview:")
print(df.head())

print("\nAnomaly distribution:")
print(df['is_anomaly'].value_counts())

df_processed, feature_columns, scaler, le_log_type, le_service = preprocess_data(df)

normal_data = df_processed[df_processed['is_anomaly'] == 0][feature_columns]
anomalous_data = df_processed[df_processed['is_anomaly'] == 1][feature_columns]

X_train, X_test = train_test_split(normal_data, test_size=0.2, random_state=42)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Anomalous samples: {anomalous_data.shape[0]}")

autoencoder, encoder = build_autoencoder(input_dim=len(feature_columns))

autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print("\nModel architecture:")
autoencoder.summary()

early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("\nTraining autoencoder...")
history = autoencoder.fit(
    X_train, X_train,
    epochs=100,
    batch_size=32,
    validation_data=(X_test, X_test),
    callbacks=[early_stopping],
    verbose=1
)

train_predictions = autoencoder.predict(X_train)
train_mse = np.mean(np.power(X_train - train_predictions, 2), axis=1)

test_predictions = autoencoder.predict(X_test)
test_mse = np.mean(np.power(X_test - test_predictions, 2), axis=1)

anomaly_predictions = autoencoder.predict(anomalous_data)
anomaly_mse = np.mean(np.power(anomalous_data - anomaly_predictions, 2), axis=1)

threshold = np.percentile(train_mse, 95)
print(f"\nAnomaly threshold (95th percentile of training MSE): {threshold:.4f}")

plt.figure(figsize=(15, 10))

plt.subplot(2, 3, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Training History')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(2, 3, 2)
plt.hist(train_mse, bins=50, alpha=0.7, label='Normal (Train)', density=True)
plt.hist(test_mse, bins=50, alpha=0.7, label='Normal (Test)', density=True)
plt.hist(anomaly_mse, bins=50, alpha=0.7, label='Anomalies', density=True)
plt.axvline(threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.4f}')
plt.xlabel('Reconstruction Error (MSE)')
plt.ylabel('Density')
plt.title('Reconstruction Error Distribution')
plt.legend()
plt.yscale('log')

plt.subplot(2, 3, 3)
all_mse = np.concatenate([test_mse, anomaly_mse])
all_labels = np.concatenate([np.zeros(len(test_mse)), np.ones(len(anomaly_mse))])
predicted_labels = (all_mse > threshold).astype(int)

cm = confusion_matrix(all_labels, predicted_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.subplot(2, 3, 4)
plt.scatter(range(len(test_mse)), test_mse, alpha=0.6, label='Normal', s=20)
plt.scatter(range(len(test_mse), len(test_mse) + len(anomaly_mse)),
           anomaly_mse, alpha=0.6, label='Anomalies', s=20)
plt.axhline(threshold, color='red', linestyle='--', label=f'Threshold: {threshold:.4f}')
plt.xlabel('Sample Index')
plt.ylabel('Reconstruction Error')
plt.title('Reconstruction Error by Sample')
plt.legend()
plt.yscale('log')

plt.subplot(2, 3, 5)
feature_importance = np.mean(np.abs(X_train - train_predictions), axis=0)
plt.bar(range(len(feature_columns)), feature_importance)
plt.title('Average Reconstruction Error by Feature')
plt.xlabel('Feature Index')
plt.ylabel('Average Absolute Error')
plt.xticks(range(len(feature_columns)), feature_columns, rotation=45)

plt.subplot(2, 3, 6)
encoded_normal = encoder.predict(X_test[:1000])
encoded_anomaly = encoder.predict(anomalous_data[:200])
plt.scatter(encoded_normal[:, 0], encoded_normal[:, 1], alpha=0.6, label='Normal', s=20)
plt.scatter(encoded_anomaly[:, 0], encoded_anomaly[:, 1], alpha=0.6, label='Anomalies', s=20)
plt.title('Encoded Representation (First 2 Dimensions)')
plt.xlabel('Encoded Dimension 1')
plt.ylabel('Encoded Dimension 2')
plt.legend()

plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(all_labels, predicted_labels,
                          target_names=['Normal', 'Anomaly']))

normal_correct = np.sum((test_mse <= threshold))
normal_total = len(test_mse)
anomaly_correct = np.sum((anomaly_mse > threshold))
anomaly_total = len(anomaly_mse)

print(f"\nDetailed Results:")
print(f"Normal samples correctly classified: {normal_correct}/{normal_total} ({100*normal_correct/normal_total:.1f}%)")
print(f"Anomalous samples correctly detected: {anomaly_correct}/{anomaly_total} ({100*anomaly_correct/anomaly_total:.1f}%)")
print(f"False positive rate: {100*(normal_total-normal_correct)/normal_total:.1f}%")
print(f"False negative rate: {100*(anomaly_total-anomaly_correct)/anomaly_total:.1f}%")

def detect_anomalies_in_new_data(new_data, autoencoder, scaler, threshold):
    new_data_scaled = scaler.transform(new_data)
    predictions = autoencoder.predict(new_data_scaled)
    mse = np.mean(np.power(new_data_scaled - predictions, 2), axis=1)
    is_anomaly = mse > threshold
    return is_anomaly, mse

print("\nTesting on new samples...")
test_samples = df_processed.sample(10)[feature_columns]
test_labels = df_processed.sample(10)['is_anomaly'].values

predictions, mse_scores = detect_anomalies_in_new_data(test_samples, autoencoder, scaler, threshold)

for i, (pred, actual, score) in enumerate(zip(predictions, test_labels, mse_scores)):
    status = "✓" if pred == actual else "✗"
    print(f"Sample {i+1}: Predicted={'Anomaly' if pred else 'Normal'}, "
          f"Actual={'Anomaly' if actual else 'Normal'}, "
          f"MSE={score:.4f} {status}")

print(f"\nSaving model components...")
autoencoder.save('log_anomaly_autoencoder.h5')
print("Model saved as 'log_anomaly_autoencoder.h5'")
print(f"Threshold: {threshold}")
print("Preprocessing components saved in memory for future use")